In [3]:
import pandas as pd
from pathlib import Path

In [5]:
DATA_DIR = Path('../data/')
MAN_DIR = DATA_DIR / 'manifests'
RAW_DIR = DATA_DIR / 'raw'
XLSX_PATH = RAW_DIR / 'pat_info.xlsx'
PAR_PATH = MAN_DIR / 'image_manifest_with_splits.parquet'

In [9]:
df = pd.read_parquet(PAR_PATH)

image_df = df[df["modality"] == "image"].copy()

image_df = image_df[
    ["patient_id", "pose_index", "filepath", "hb_grade", "split"]
].copy()

,patient_id,pose_index,filepath,hb_grade,split
0,CompleteFlaccid1,1.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
1,CompleteFlaccid1,2.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
2,CompleteFlaccid1,3.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
3,CompleteFlaccid1,4.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
4,CompleteFlaccid1,5.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
...,...,...,...,...,...
534,SevereSynkinetic5,4.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,5,test
535,SevereSynkinetic5,5.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,5,test
536,SevereSynkinetic5,6.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,5,test
537,SevereSynkinetic5,7.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,5,test


In [11]:
image_df = image_df.sort_values(
    by=["patient_id", "pose_index"]
).reset_index(drop=True)

In [13]:
print("Image-only table shape:", image_df.shape)
print("Unique patients:", image_df["patient_id"].nunique())
image_df.head(10)

Image-only table shape: (480, 5)
Unique patients: 60


,patient_id,pose_index,filepath,hb_grade,split
0,CompleteFlaccid1,1.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
1,CompleteFlaccid1,2.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
2,CompleteFlaccid1,3.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
3,CompleteFlaccid1,4.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
4,CompleteFlaccid1,5.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
5,CompleteFlaccid1,6.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
6,CompleteFlaccid1,7.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
7,CompleteFlaccid1,8.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,val
8,CompleteFlaccid2,1.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,train
9,CompleteFlaccid2,2.0,/Users/willcrosswhite/Documents/All_Code/Palsy...,6,train


In [18]:
pose_counts = (
    image_df.groupby("patient_id")
    .size()
    .reset_index(name="num_images")
)

print(pose_counts["num_images"].value_counts().sort_index())
pose_counts.head

num_images
8    60
Name: count, dtype: int64


<bound method NDFrame.head of                patient_id  num_images
0        CompleteFlaccid1           8
1        CompleteFlaccid2           8
2        CompleteFlaccid3           8
3        CompleteFlaccid4           8
4        CompleteFlaccid5           8
5     CompleteSynkinetic1           8
6     CompleteSynkinetic2           8
7     CompleteSynkinetic3           8
8     CompleteSynkinetic4           8
9     CompleteSynkinetic5           8
10           MildFlaccid1           8
11           MildFlaccid2           8
12           MildFlaccid3           8
13           MildFlaccid4           8
14           MildFlaccid5           8
15        MildSynkinetic1           8
16        MildSynkinetic2           8
17        MildSynkinetic3           8
18        MildSynkinetic4           8
19        MildSynkinetic5           8
20       ModerateFlaccid1           8
21       ModerateFlaccid2           8
22       ModerateFlaccid3           8
23       ModerateFlaccid4           8
24       ModerateFla

In [19]:
expected_num_poses = 8

incomplete_patients = pose_counts[pose_counts["num_images"] != expected_num_poses].copy()

print("Number of incomplete patients:", len(incomplete_patients))
print(incomplete_patients)

Number of incomplete patients: 0
Empty DataFrame
Columns: [patient_id, num_images]
Index: []


In [20]:
pose_lists = (
    image_df.groupby("patient_id")["pose_index"]
    .apply(lambda s: sorted(s.tolist()))
    .reset_index(name="pose_indices")
)

print(pose_lists.head())

         patient_id                              pose_indices
0  CompleteFlaccid1  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
1  CompleteFlaccid2  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
2  CompleteFlaccid3  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
3  CompleteFlaccid4  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
4  CompleteFlaccid5  [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]


In [21]:
expected_pose_set = set(range(1, 9))

def find_missing_poses(pose_series):
    observed = set(int(x) for x in pose_series if pd.notna(x))
    missing = sorted(expected_pose_set - observed)
    extra = sorted(observed - expected_pose_set)
    return pd.Series({
        "missing_poses": missing,
        "extra_poses": extra,
        "is_complete": observed == expected_pose_set
    })

pose_audit = (
    image_df.groupby("patient_id")["pose_index"]
    .apply(find_missing_poses)
    .reset_index()
)

print(pose_audit.head())

         patient_id        level_1 pose_index
0  CompleteFlaccid1  missing_poses         []
1  CompleteFlaccid1    extra_poses         []
2  CompleteFlaccid1    is_complete       True
3  CompleteFlaccid2  missing_poses         []
4  CompleteFlaccid2    extra_poses         []
